# 🏥 AI Medical Assistant — Text Models (LSTM + GRU + BERT)
### Notebook كامل: Data → LSTM → GRU → BERT → Comparison


## 📦 Step 1 — Imports & Setup

In [ ]:
import requests, time, re, pickle, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings('ignore')
print("✅ TensorFlow:", tf.__version__)
print("✅ All imports done!")


## 🌐 Step 2 — Data Collection (FDA API)

In [ ]:
API_KEY  = "310dfba3252d94966cb65a063f8174058108"
BASE_URL = "https://api.fda.gov/drug/label.json"

drug_terms = [
    "ibuprofen", "aspirin", "paracetamol",
    "metformin", "amoxicillin", "insulin",
    "omeprazole", "atorvastatin", "azithromycin",
    "cetirizine", "diclofenac", "prednisone", "salbutamol"
]
print(f"📋 {len(drug_terms)} drugs:", drug_terms)


In [ ]:
def fetch_medical_data(terms, target_size=2000):
    all_data = []
    print("🚀 Collecting data...")
    while len(all_data) < target_size:
        for term in tqdm(terms, desc="Fetching"):
            url = f"{BASE_URL}?search=openfda.generic_name:{term}&limit=100"
            r   = requests.get(url)
            if r.status_code != 200:
                continue
            for item in r.json().get("results", []):
                all_data.append({
                    "drug"    : term,
                    "text"    : " ".join(item.get("description", [""])),
                    "purpose" : " ".join(item.get("purpose",     [""])),
                    "warnings": " ".join(item.get("warnings",    [""]))
                })
                if len(all_data) >= target_size:
                    break
            time.sleep(0.2)
            if len(all_data) >= target_size:
                break
    df = pd.DataFrame(all_data)
    print(f"\n✅ Collected: {len(df)} records")
    print(df["drug"].value_counts())
    return df

df = fetch_medical_data(drug_terms, target_size=2000)
df.head(3)


## 🧹 Step 3 — Text Cleaning & Preprocessing

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+",         " ", text)
    return text.strip()

df["clean_text"] = (
    df["text"].fillna("") + " " +
    df["purpose"].fillna("") + " " +
    df["warnings"].fillna("")
)
df["clean_text"] = df["clean_text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 10].reset_index(drop=True)

print(f"✅ After cleaning: {len(df)} records")
print("\nSample:")
print(df["clean_text"].iloc[0][:300])


## 🏷️ Step 4 — Encoding & Tokenization

In [ ]:
VOCAB_SIZE    = 5000
MAX_LEN       = 120
EMBEDDING_DIM = 64

# Label encoding
le = LabelEncoder()
df["label"] = le.fit_transform(df["drug"])
NUM_CLASSES  = len(le.classes_)
print(f"✅ {NUM_CLASSES} classes:", list(le.classes_))

# Tokenize
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df["clean_text"])
sequences = tokenizer.texts_to_sequences(df["clean_text"])

X = pad_sequences(sequences, maxlen=MAX_LEN, padding="post", truncating="post")
y = to_categorical(df["label"], num_classes=NUM_CLASSES)

print(f"\n✅ X shape: {X.shape}")
print(f"✅ y shape: {y.shape}")


## ✂️ Step 5 — Train / Val / Test Split  (70 / 15 / 15)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Train : {X_train.shape[0]}")
print(f"Val   : {X_val.shape[0]}")
print(f"Test  : {X_test.shape[0]}")


## 🧠 Step 6 — LSTM Baseline Model
> Bidirectional LSTM يقرأ النص forward وbackward للحصول على context أفضل


In [ ]:
def build_lstm(vocab_size, embedding_dim, max_len, num_classes):
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_len, name="embedding"),
        Bidirectional(LSTM(128, return_sequences=True), name="biLSTM_1"),
        Dropout(0.3),
        LSTM(64, name="LSTM_2"),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dropout(0.2),
        Dense(num_classes, activation="softmax", name="output")
    ], name="LSTM_Drug_Classifier")

    model.compile(optimizer="adam",
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model

lstm_model = build_lstm(VOCAB_SIZE, EMBEDDING_DIM, MAX_LEN, NUM_CLASSES)
lstm_model.summary()


## 🏋️ Step 7 — Train LSTM

In [ ]:
lstm_callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=5,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint("best_lstm.h5", monitor="val_accuracy",
                    save_best_only=True, verbose=1)
]

lstm_history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=lstm_callbacks
)
print("\n✅ LSTM Training complete!")


## 📈 Step 8 — Plot LSTM Curves

In [ ]:
def plot_history(history, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(history.history["accuracy"],     label="Train", color="royalblue")
    axes[0].plot(history.history["val_accuracy"], label="Val",   color="tomato")
    axes[0].set_title(f"Accuracy — {title}"); axes[0].legend(); axes[0].grid(alpha=.3)

    axes[1].plot(history.history["loss"],     label="Train", color="royalblue")
    axes[1].plot(history.history["val_loss"], label="Val",   color="tomato")
    axes[1].set_title(f"Loss — {title}"); axes[1].legend(); axes[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

plot_history(lstm_history, "LSTM Baseline")


## 📊 Step 9 — Evaluate LSTM

In [ ]:
def evaluate_model(model, X_test, y_test, label_encoder, model_name="Model"):
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    y_pred     = model.predict(X_test, verbose=0)
    y_pred_cls = np.argmax(y_pred, axis=1)
    y_true_cls = np.argmax(y_test, axis=1)

    print(f"\n{'='*50}")
    print(f"📊 {model_name} Results")
    print(f"{'='*50}")
    print(f"🎯 Test Accuracy : {acc*100:.2f}%")
    print(f"📉 Test Loss     : {loss:.4f}")
    print(f"\n{classification_report(y_true_cls, y_pred_cls, target_names=label_encoder.classes_)}")

    # Confusion matrix
    cm = confusion_matrix(y_true_cls, y_pred_cls)
    plt.figure(figsize=(11, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title(f"Confusion Matrix — {model_name}", fontsize=13)
    plt.ylabel("True"); plt.xlabel("Predicted")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()
    return acc

lstm_acc = evaluate_model(lstm_model, X_test, y_test, le, "LSTM Baseline")


## 🟣 Step 10 — GRU Model
> Bidirectional GRU — أسرع من LSTM، نفس الفكرة بـ gates أقل

In [ ]:
from tensorflow.keras.layers import GRU

def build_gru(vocab_size, embedding_dim, max_len, num_classes):
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_len),
        Bidirectional(GRU(128, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(GRU(64)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ], name='GRU_Drug_Classifier')

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

gru_model = build_gru(VOCAB_SIZE, EMBEDDING_DIM, MAX_LEN, NUM_CLASSES)
gru_model.summary()


## 🏋️ Step 11 — Train GRU

In [ ]:
gru_history = gru_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=5,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint('best_gru.h5', monitor='val_accuracy',
                        save_best_only=True, verbose=1)
    ]
)
print("\n✅ GRU Training complete!")


## 📈 Step 12 — Plot GRU Curves

In [ ]:
plot_history(gru_history, "GRU")


## 📊 Step 13 — Evaluate GRU

In [ ]:
gru_acc = evaluate_model(gru_model, X_test, y_test, le, "GRU")


## 🔵 Step 14 — AraBERT / BERT (Advanced Text Model)
> نستخدم `bert-base-uncased` هنا لأن الداتا English  
> لو عندك داتا عربية → غيّر لـ `aubmindlab/bert-base-arabertv02`


In [ ]:
# Install transformers (Colab already has it usually)
!pip install transformers -q
print("✅ transformers ready")


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {DEVICE}")

# ── Choose model ──
# English data  → "bert-base-uncased"
# Arabic data   → "aubmindlab/bert-base-arabertv02"
BERT_MODEL_NAME = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
print(f"✅ Loaded tokenizer: {BERT_MODEL_NAME}")


In [ ]:
class DrugDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label"         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Prepare raw labels (not one-hot)
y_raw = df["label"].values
X_raw = df["clean_text"].values

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_raw, y_raw, test_size=0.30, random_state=42, stratify=y_raw)
X_v,  X_te, y_v,  y_te  = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

BERT_BATCH = 16
train_ds = DrugDataset(list(X_tr), list(y_tr), bert_tokenizer)
val_ds   = DrugDataset(list(X_v),  list(y_v),  bert_tokenizer)
test_ds  = DrugDataset(list(X_te), list(y_te), bert_tokenizer)

train_loader = DataLoader(train_ds, batch_size=BERT_BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BERT_BATCH)
test_loader  = DataLoader(test_ds,  batch_size=BERT_BATCH)

print(f"✅ Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")


In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME, num_labels=NUM_CLASSES
).to(DEVICE)

BERT_EPOCHS = 4
optimizer   = AdamW(bert_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * BERT_EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps//10,
    num_training_steps=total_steps)

print(f"✅ BERT model loaded — {sum(p.numel() for p in bert_model.parameters()):,} params")


In [ ]:
def train_bert_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, correct = 0, 0
    for batch in tqdm(loader, desc="Training", leave=False):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["label"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        logits  = outputs.logits

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        correct    += (logits.argmax(dim=1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

def eval_bert(model, loader):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["label"].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            correct    += (outputs.logits.argmax(dim=1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

# ── Training loop ──
bert_train_accs, bert_val_accs = [], []
best_val_acc = 0

for epoch in range(BERT_EPOCHS):
    tr_loss, tr_acc = train_bert_epoch(bert_model, train_loader, optimizer, scheduler)
    vl_loss, vl_acc = eval_bert(bert_model, val_loader)
    bert_train_accs.append(tr_acc); bert_val_accs.append(vl_acc)
    print(f"Epoch {epoch+1}/{BERT_EPOCHS} | Train Acc: {tr_acc:.4f} | Val Acc: {vl_acc:.4f}")
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(bert_model.state_dict(), "best_bert.pt")
        print("  💾 Saved best model")

print("\n✅ BERT training complete!")


## 📊 Step 15 — Evaluate BERT

In [ ]:
# Load best weights
bert_model.load_state_dict(torch.load("best_bert.pt"))
bert_model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["label"].to(DEVICE)
        logits = bert_model(input_ids=input_ids, attention_mask=attention_mask).logits
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_true.extend(labels.cpu().numpy())

bert_acc = accuracy_score(all_true, all_preds)
print(f"\n{'='*50}")
print(f"📊 BERT / AraBERT Results")
print(f"{'='*50}")
print(f"🎯 Test Accuracy : {bert_acc*100:.2f}%")
print(f"\n{classification_report(all_true, all_preds, target_names=le.classes_)}")

cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(11, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix — BERT", fontsize=13)
plt.ylabel("True"); plt.xlabel("Predicted")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()


## ⚖️ Step 16 — Model Comparison

In [ ]:
# BERT training curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, BERT_EPOCHS+1), bert_train_accs, "o-", label="BERT Train", color="orange")
ax.plot(range(1, BERT_EPOCHS+1), bert_val_accs,   "s--",label="BERT Val",   color="tomato")
ax.set_title("BERT Accuracy per Epoch"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
# Side-by-side bar chart
models = ["LSTM Baseline", "BERT / AraBERT"]
accs   = [lstm_acc * 100, bert_acc * 100]

colors = ["royalblue", "darkorange"]
plt.figure(figsize=(7, 4))
bars = plt.bar(models, accs, color=colors, edgecolor="white", width=0.4)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{acc:.1f}%", ha="center", fontsize=13, fontweight="bold")
plt.ylim(0, 110)
plt.title("Model Comparison — Test Accuracy", fontsize=14)
plt.ylabel("Accuracy (%)")
plt.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()

print("\n📋 Summary:")
print(f"   LSTM Baseline : {lstm_acc*100:.2f}%")
print(f"   BERT/AraBERT  : {bert_acc*100:.2f}%")
winner = "BERT/AraBERT" if bert_acc > lstm_acc else "LSTM Baseline"
print(f"\n🏆 Winner: {winner}")


## 🔮 Step 17 — Unified Predict Function

In [ ]:
def predict_drug_lstm(text, top_k=3):
    cleaned = clean_text(text)
    seq     = tokenizer.texts_to_sequences([cleaned])
    padded  = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    probs   = lstm_model.predict(padded, verbose=0)[0]
    top_idx = np.argsort(probs)[::-1][:top_k]
    print(f"\n[LSTM] 🔍 '{text[:60]}'")
    for i, idx in enumerate(top_idx, 1):
        bar = "█" * int(probs[idx]*100/5)
        print(f"  {i}. {le.classes_[idx]:<16} {probs[idx]*100:>6.2f}%  {bar}")
    return le.classes_[top_idx[0]]

def predict_drug_bert(text, top_k=3):
    bert_model.eval()
    enc = bert_tokenizer(text, max_length=128, padding="max_length",
                         truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = bert_model(
            input_ids=enc["input_ids"].to(DEVICE),
            attention_mask=enc["attention_mask"].to(DEVICE)
        ).logits
    probs   = torch.softmax(logits, dim=1)[0].cpu().numpy()
    top_idx = np.argsort(probs)[::-1][:top_k]
    print(f"\n[BERT] 🔍 '{text[:60]}'")
    for i, idx in enumerate(top_idx, 1):
        bar = "█" * int(probs[idx]*100/5)
        print(f"  {i}. {le.classes_[idx]:<16} {probs[idx]*100:>6.2f}%  {bar}")
    return le.classes_[top_idx[0]]

# ── Test ──
sample = "used for pain relief fever reduction anti inflammatory"
predict_drug_lstm(sample)
predict_drug_bert(sample)


## 💾 Step 18 — Save All Models

In [ ]:
# LSTM
lstm_model.save("lstm_drug_classifier.h5")
with open("tokenizer.pkl", "wb") as f: pickle.dump(tokenizer, f)
with open("label_encoder.pkl", "wb") as f: pickle.dump(le, f)

# BERT best weights already saved as best_bert.pt
print("✅ lstm_drug_classifier.h5")
print("✅ tokenizer.pkl")
print("✅ label_encoder.pkl")
print("✅ best_bert.pt")
print("\n🎉 Text Models Done!")
print("   Next step → Image Model (ResNet50) 🖼️")
